In [1]:
import pandas as pd
import numpy as np

# Load processed dataset
df = pd.read_csv("../data/processed/mta_subway_understood.csv")

print("Shape:", df.shape)
print(df.head())

Shape: (500000, 17)
     transit_timestamp transit_mode station_complex_id  \
0  2022-09-27 07:00:00       subway                620   
1  2022-09-27 03:00:00       subway                636   
2  2022-09-27 08:00:00       subway                428   
3  2022-09-27 13:00:00       subway                 79   
4  2022-09-27 07:00:00       subway                118   

                       station_complex    borough payment_method  \
0  Court St (R)/Borough Hall (2,3,4,5)   Brooklyn      metrocard   
1           Jay St-MetroTech (A,C,F,R)   Brooklyn      metrocard   
2                         174 St (2,5)      Bronx      metrocard   
3                            86 St (N)   Brooklyn           omny   
4                             3 Av (L)  Manhattan      metrocard   

                fare_class_category  ridership  transfers   latitude  \
0       Metrocard - Unlimited 7-Day       30.0        0.0  40.693220   
1                 Metrocard - Other        4.0        0.0  40.692337   
2     

/var/folders/tq/4kznmln90v59l2b_8bd4y7xr0000gn/T/ipykernel_3793/1348883503.py:5: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/mta_subway_understood.csv")


In [2]:
print("Columns:")
print(df.columns.tolist())

Columns:
['transit_timestamp', 'transit_mode', 'station_complex_id', 'station_complex', 'borough', 'payment_method', 'fare_class_category', 'ridership', 'transfers', 'latitude', 'longitude', 'georeference', 'hour', 'day', 'month', 'day_of_week', 'is_weekend']


In [3]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
transit_timestamp      0
transit_mode           0
station_complex_id     0
station_complex        0
borough                0
payment_method         0
fare_class_category    0
ridership              0
transfers              0
latitude               0
longitude              0
georeference           0
hour                   0
day                    0
month                  0
day_of_week            0
is_weekend             0
dtype: int64


In [4]:
# Drop columns not needed for prediction
df = df.drop(columns=["georeference"], errors="ignore")

print("Remaining columns:")
print(df.columns.tolist())

Remaining columns:
['transit_timestamp', 'transit_mode', 'station_complex_id', 'station_complex', 'borough', 'payment_method', 'fare_class_category', 'ridership', 'transfers', 'latitude', 'longitude', 'hour', 'day', 'month', 'day_of_week', 'is_weekend']


In [5]:
# Fill missing numeric values with median
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill missing categorical values with mode
categorical_cols = df.select_dtypes(include="object").columns
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after handling:")
print(df.isnull().sum())

Missing values after handling:
transit_timestamp      0
transit_mode           0
station_complex_id     0
station_complex        0
borough                0
payment_method         0
fare_class_category    0
ridership              0
transfers              0
latitude               0
longitude              0
hour                   0
day                    0
month                  0
day_of_week            0
is_weekend             0
dtype: int64


In [6]:
# Convert timestamp again just in case
df["transit_timestamp"] = pd.to_datetime(df["transit_timestamp"], errors="coerce")

# Drop rows where timestamp failed
df = df.dropna(subset=["transit_timestamp"])

print("Shape after timestamp cleaning:", df.shape)

Shape after timestamp cleaning: (500000, 16)


In [7]:
# We already created time-based columns earlier, so keep useful features
# Target variable
y = df["ridership"]

# Drop target and raw timestamp
X = df.drop(columns=["ridership", "transit_timestamp"], errors="ignore")

print("X shape:", X.shape)
print("y shape:", y.shape)
print(X.head())

X shape: (500000, 14)
y shape: (500000,)
  transit_mode station_complex_id                      station_complex  \
0       subway                620  Court St (R)/Borough Hall (2,3,4,5)   
1       subway                636           Jay St-MetroTech (A,C,F,R)   
2       subway                428                         174 St (2,5)   
3       subway                 79                            86 St (N)   
4       subway                118                             3 Av (L)   

     borough payment_method               fare_class_category  transfers  \
0   Brooklyn      metrocard       Metrocard - Unlimited 7-Day        0.0   
1   Brooklyn      metrocard                 Metrocard - Other        0.0   
2      Bronx      metrocard      Metrocard - Unlimited 30-Day        0.0   
3   Brooklyn           omny                  OMNY - Full Fare        1.0   
4  Manhattan      metrocard  Metrocard - Seniors & Disability        0.0   

    latitude  longitude  hour  day  month  day_of_week  i

In [8]:
# One-hot encode categorical columns
X = pd.get_dummies(X, drop_first=True)

print("Shape after encoding:", X.shape)
print(X.head())

Shape after encoding: (500000, 1304)
   transfers   latitude  longitude  hour  day  month  day_of_week  is_weekend  \
0        0.0  40.693220  -73.99000     7   27      9            1           0   
1        0.0  40.692337  -73.98734     3   27      9            1           0   
2        0.0  40.837288  -73.88773     8   27      9            1           0   
3        1.0  40.592720  -73.97823    13   27      9            1           0   
4        0.0  40.732850  -73.98612     7   27      9            1           0   

   transit_mode_subway  transit_mode_tram  ...  payment_method_omny  \
0                 True              False  ...                False   
1                 True              False  ...                False   
2                 True              False  ...                False   
3                 True              False  ...                 True   
4                 True              False  ...                False   

   fare_class_category_Metrocard - Full Fare  \
0

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (400000, 1304)
X_test: (100000, 1304)
y_train: (400000,)
y_test: (100000,)


In [10]:
# Save preprocessed files for model training
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Preprocessed train-test files saved successfully!")

Preprocessed train-test files saved successfully!
